# House Prices - ElasticNet Regularization

This notebook extends the previous Ridge and Lasso experiments.

Goal:
- keep the same polynomial feature representation
- test ElasticNet regularization
- tune both `alpha` and `l1_ratio` with 5-fold cross-validation
- compare ElasticNet against the previous Ridge and Lasso benchmarks
- generate a new Kaggle submission if performance improves

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold, cross_val_score

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

X = train_df.drop("SalePrice", axis=1)
y_log = np.log(train_df["SalePrice"])
X_test = test_df.copy()

print("X shape:", X.shape)
print("y_log shape:", y_log.shape)
print("X_test shape:", X_test.shape)

X shape: (1460, 80)
y_log shape: (1460,)
X_test shape: (1459, 80)


In [2]:
numeric_features = X.select_dtypes(include=["number"]).columns.drop("Id")
categorical_features = X.select_dtypes(exclude=["number"]).columns

poly_features = ["GrLivArea", "OverallQual", "TotalBsmtSF", "GarageArea"]

remaining_numeric_features = [
    col for col in numeric_features if col not in poly_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Polynomial features:", poly_features)
print("Remaining numeric features:", len(remaining_numeric_features))

Numeric features: 36
Categorical features: 43
Polynomial features: ['GrLivArea', 'OverallQual', 'TotalBsmtSF', 'GarageArea']
Remaining numeric features: 32


In [3]:
poly_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

remaining_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_poly = ColumnTransformer(transformers=[
    ("poly_num", poly_numeric_transformer, poly_features),
    ("num", remaining_numeric_transformer, remaining_numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor ready.")

Preprocessor ready.


In [4]:
ridge_best_cv_rmse = 0.12435925780325499
lasso_best_cv_rmse = 0.12385617304987953

print("Previous Ridge benchmark mean CV RMSE:", ridge_best_cv_rmse)
print("Previous Lasso benchmark mean CV RMSE:", lasso_best_cv_rmse)

Previous Ridge benchmark mean CV RMSE: 0.12435925780325499
Previous Lasso benchmark mean CV RMSE: 0.12385617304987953


In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
elasticnet_alphas = [0.0001, 0.0003, 0.001, 0.003, 0.01]
elasticnet_l1_ratios = [0.2, 0.4, 0.6, 0.8, 0.9]

elasticnet_results = []

for alpha in elasticnet_alphas:
    for l1_ratio in elasticnet_l1_ratios:
        elasticnet_model = Pipeline(steps=[
            ("preprocessor", preprocessor_poly),
            ("model", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000))
        ])

        scores = cross_val_score(
            elasticnet_model,
            X,
            y_log,
            cv=kf,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        rmse_scores = -scores

        elasticnet_results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "mean_cv_rmse": rmse_scores.mean(),
            "std_cv_rmse": rmse_scores.std()
        })

elasticnet_results_df = pd.DataFrame(elasticnet_results).sort_values("mean_cv_rmse")
elasticnet_results_df.head(10)

,alpha,l1_ratio,mean_cv_rmse,std_cv_rmse
10,0.0010,0.2,0.123300,0.017953
8,0.0003,0.8,0.123305,0.017863
7,0.0003,0.6,0.123317,0.016678
9,0.0003,0.9,0.123549,0.018631
6,0.0003,0.4,0.123988,0.015932
5,0.0003,0.2,0.124877,0.014729
11,0.0010,0.4,0.125707,0.023766
4,0.0001,0.9,0.125719,0.015787
3,0.0001,0.8,0.125890,0.015557
2,0.0001,0.6,0.126368,0.015056


In [7]:
best_elasticnet_row = elasticnet_results_df.iloc[0]

print("Best ElasticNet alpha:", best_elasticnet_row["alpha"])
print("Best ElasticNet l1_ratio:", best_elasticnet_row["l1_ratio"])
print("Best ElasticNet mean CV RMSE:", best_elasticnet_row["mean_cv_rmse"])
print("Best ElasticNet std CV RMSE:", best_elasticnet_row["std_cv_rmse"])
print()

print("Previous Ridge benchmark mean CV RMSE:", ridge_best_cv_rmse)
print("Previous Lasso benchmark mean CV RMSE:", lasso_best_cv_rmse)
print()

print("Difference vs Ridge:", best_elasticnet_row["mean_cv_rmse"] - ridge_best_cv_rmse)
print("Difference vs Lasso:", best_elasticnet_row["mean_cv_rmse"] - lasso_best_cv_rmse)

Best ElasticNet alpha: 0.001
Best ElasticNet l1_ratio: 0.2
Best ElasticNet mean CV RMSE: 0.12330001703822155
Best ElasticNet std CV RMSE: 0.017952722499537343

Previous Ridge benchmark mean CV RMSE: 0.12435925780325499
Previous Lasso benchmark mean CV RMSE: 0.12385617304987953

Difference vs Ridge: -0.0010592407650334423
Difference vs Lasso: -0.0005561560116579822


In [8]:
elasticnet_alphas_fine = [0.0005, 0.0007, 0.001, 0.0015, 0.002, 0.003]
elasticnet_l1_ratios_fine = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]

elasticnet_results_fine = []

for alpha in elasticnet_alphas_fine:
    for l1_ratio in elasticnet_l1_ratios_fine:
        elasticnet_model = Pipeline(steps=[
            ("preprocessor", preprocessor_poly),
            ("model", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=30000
            ))
        ])

        scores = cross_val_score(
            elasticnet_model,
            X,
            y_log,
            cv=kf,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        rmse_scores = -scores

        elasticnet_results_fine.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "mean_cv_rmse": rmse_scores.mean(),
            "std_cv_rmse": rmse_scores.std()
        })

elasticnet_results_fine_df = pd.DataFrame(elasticnet_results_fine).sort_values("mean_cv_rmse")
elasticnet_results_fine_df.head(15)

,alpha,l1_ratio,mean_cv_rmse,std_cv_rmse
6,0.0005,0.40,0.123028,0.017221
12,0.0007,0.30,0.123107,0.017713
11,0.0007,0.25,0.123168,0.017035
17,0.0010,0.20,0.123300,0.017953
16,0.0010,0.15,0.123308,0.016966
10,0.0007,0.20,0.123310,0.016470
5,0.0005,0.30,0.123350,0.016424
15,0.0010,0.10,0.123451,0.016102
9,0.0007,0.15,0.123515,0.015914
4,0.0005,0.25,0.123526,0.016065


In [9]:
best_elasticnet_fine_row = elasticnet_results_fine_df.iloc[0]

print("Best fine-tuned ElasticNet alpha:", best_elasticnet_fine_row["alpha"])
print("Best fine-tuned ElasticNet l1_ratio:", best_elasticnet_fine_row["l1_ratio"])
print("Best fine-tuned ElasticNet mean CV RMSE:", best_elasticnet_fine_row["mean_cv_rmse"])
print("Best fine-tuned ElasticNet std CV RMSE:", best_elasticnet_fine_row["std_cv_rmse"])
print()
print("Previous coarse-grid ElasticNet mean CV RMSE:", best_elasticnet_row["mean_cv_rmse"])
print("Difference:", best_elasticnet_fine_row["mean_cv_rmse"] - best_elasticnet_row["mean_cv_rmse"])

Best fine-tuned ElasticNet alpha: 0.0005
Best fine-tuned ElasticNet l1_ratio: 0.4
Best fine-tuned ElasticNet mean CV RMSE: 0.12302807178411253
Best fine-tuned ElasticNet std CV RMSE: 0.017220511285103917

Previous coarse-grid ElasticNet mean CV RMSE: 0.12330001703822155
Difference: -0.00027194525410902115


In [10]:
best_elasticnet_alpha = float(best_elasticnet_fine_row["alpha"])
best_elasticnet_l1_ratio = float(best_elasticnet_fine_row["l1_ratio"])

best_elasticnet_model = Pipeline(steps=[
    ("preprocessor", preprocessor_poly),
    ("model", ElasticNet(
        alpha=best_elasticnet_alpha,
        l1_ratio=best_elasticnet_l1_ratio,
        max_iter=30000
    ))
])

best_elasticnet_model.fit(X, y_log)

print("Best fine-tuned ElasticNet final model fitted on full training set.")
print("alpha =", best_elasticnet_alpha)
print("l1_ratio =", best_elasticnet_l1_ratio)

Best fine-tuned ElasticNet final model fitted on full training set.
alpha = 0.0005
l1_ratio = 0.4


In [11]:
test_pred_log_elasticnet = best_elasticnet_model.predict(X_test)
test_pred_elasticnet = np.exp(test_pred_log_elasticnet)

print("Number of predictions:", len(test_pred_elasticnet))
print("First 10 predicted prices:")
print(test_pred_elasticnet[:10])

Number of predictions: 1459
First 10 predicted prices:
[118009.52657292 154981.79671013 180756.40937707 199021.00014361
 196134.20454899 172101.14156127 183943.71809023 160620.92067454
 196049.39974698 119932.29799513]


In [12]:
submission_elasticnet = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred_elasticnet
})

print(submission_elasticnet.shape)
submission_elasticnet.head()

(1459, 2)


,Id,SalePrice
0,1461,118009.526573
1,1462,154981.796710
2,1463,180756.409377
3,1464,199021.000144
4,1465,196134.204549


In [13]:
submission_elasticnet_path = "../submissions/submission_elasticnet_alpha_0_0005_l1ratio_0_4_cv.csv"

submission_elasticnet.to_csv(submission_elasticnet_path, index=False)

print("Submission saved to:", submission_elasticnet_path)

Submission saved to: ../submissions/submission_elasticnet_alpha_0_0005_l1ratio_0_4_cv.csv
